<a href="https://colab.research.google.com/github/Radiat09/colab/blob/main/InClass_of_Boston_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Boston Housing EDA: A Friendly Survival Guide

হ্যালো প্রোগ্রামারস! **Week 01 Conceptual Session**-এর কাজ হিসেবে আমরা **Pandas + EDA** করবো — টেবিল খুলে দেখব, মিসিং ধরব, চার্ট বানাব।

Dataset: [Boston House Prices on Kaggle](https://www.kaggle.com/datasets/vikrishnan/boston-house-prices)  
File: `housing.csv` (506 rows, 14 columns, whitespace-separated, **no header row**).

This is **Exploratory Data Analysis**, not "fit a model and buy a house in Boston."

> "You cannot negotiate rent with `plt.scatter`. But EDA tells you which features actually matter before the model lies politely."

**How to use this:**
- Run top to bottom. Plots reuse `df`.
- Read the joke. Then look at the chart.
- Practice at the end is empty on purpose.

**Map**
1. What is EDA  
2. Analytical view of a dataset  
3. Input screening  
4. Statistical profiling  
5. Univariate — categorical (`CHAS`)  
6. Univariate — numerical (`RM`, `LSTAT`, `MEDV`)  
7. Countplot with hue  
8. Barplot (category vs number)  
9. KDE / density  
10. Scatterplot  
11. Y profiling (`MEDV`)  
12. Practice  


In [ ]:
# !pip install pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

BOSTON_COLS = [
    "CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS",
    "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV",
]

candidates = [
    Path("/content/housing.csv"),
    Path("data/housing.csv"),
]
DATA = next((p for p in candidates if p.exists()), None)
if DATA is None:
    raise FileNotFoundError(
        "Download housing.csv from Kaggle and put it next to this notebook."
    )

df = pd.read_csv(DATA, sep=r'\s+', names=BOSTON_COLS, header=None)
print("loaded:", DATA)
print("shape :", df.shape)
print("Good. 506 neighborhoods. Still cheaper than 2026 rent jokes.")


loaded: /content/housing.csv
shape : (506, 14)
Good. 506 neighborhoods. Still cheaper than 2026 rent jokes.


In [ ]:
#

---

# 1. What is EDA

**EDA** = look at the table before you marry it.

Questions EDA answers cheaply:

- How many rows? What columns?
- What is the **target** (`MEDV` = median home value in $1000s)?
- Which columns are numbers vs categories?
- Any missing values?
- Does river view (`CHAS`) or room count (`RM`) shout in the plots?

Skip EDA and the model will still run. It might learn that row 47 is special because it is row 47.


**Target (Y):** `MEDV` — median house price in **thousands of dollars**.

Example: `MEDV = 24.0` means about **$24,000** (dataset is old; do not move to Boston with this budget).

**Categorical (for us today):** `CHAS` (0/1 = not on Charles River / on river).  
**Numeric stars:** `RM` (rooms), `LSTAT` (% lower status), `NOX`, `AGE`, etc.


# 2. Analytical view of a dataset

First date: `head`, `columns`, `shape`. Do not print 506 rows like homework revenge.


In [ ]:
print("Rows, Clos:",df.shape)

Rows, Clos: (506, 14)


In [ ]:
df

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,393.45,6.48,22.0


### Column cheat sheet (short)

| Col | Meaning |
|-----|---------|
| `CRIM` | crime rate by town |
| `RM` | avg number of rooms |
| `AGE` | old homes proportion |
| `DIS` | distance to employment centers |
| `CHAS` | Charles River dummy (0/1) |
| `LSTAT` | % lower status population |
| `MEDV` | **target** — median home value ($1000s) |
| ... | rest are town/industry/tax vibes |

Full names are in the sklearn docs. For EDA, know **target**, **CHAS**, **RM**, **LSTAT**. That is enough to start.


# 3. Input screening

Check dtypes, missing values, duplicates, weird ranges.


In [ ]:
df.duplicated().sum()

np.int64(0)

Boston housing in this file: usually **no missing**. If your copy has holes, your download fought someone.

`CHAS` should be only 0 or 1. If you see 2, something imported wrong.


In [ ]:
df["CHAS"].value_counts()

,count
CHAS,
0,471
1,35


# 4. Statistical profiling

`describe()` for numbers. This is the table clearing its throat.


**Read it like a human:**

- `RM` mean ~6 rooms. Not a mansion. Not a shoebox.
- `LSTAT` spread matters — inequality shows up in house prices later.
- `MEDV` max 50 → $50k cap in this dataset. Inflation was not invited.


# 5. Univariate — categorical (`CHAS`)

Univariate = one column. `CHAS`: river or no river. Only 35 river towns. Rare flex.


# 6. Univariate — numerical

`RM`, `LSTAT`, and target `MEDV`. Histogram + boxplot = where values live and who is an outlier.


`LSTAT` and `MEDV` often look skewed. Real life piles up at one end. Mean is dramatic. Median is honest.


# 7. Countplot with hue

Make a cheap **price tag**: above-median `MEDV` = pricey neighborhood (for this dataset).

```python
df["Pricey"] = (df["MEDV"] > df["MEDV"].median()).astype(int)
```

Then: river towns vs not, split by pricey or not.


# 8. Barplot — category vs number

**Barplot** = average of a numeric column per category.

Does river view (`CHAS`) get higher average `MEDV`? Does crime (`CRIM`) track with price if we bin towns? Start simple: mean `MEDV` by `CHAS`.


# 9. KDE — density

KDE = smooth histogram. Compare **where** values pile up.

- `LSTAT` for river vs not (`hue=CHAS`)
- `RM` for pricey vs not


# 10. Scatterplot

Classic pair: **`RM` vs `LSTAT`**. Color by `MEDV` or `CHAS`.

More rooms + lower LSTAT often means pricier. EDA checks if the cloud agrees.


# 11. Y profiling (`MEDV`)

Profile the target:

- distribution (histogram)
- is it skewed?
- mean `MEDV` by `CHAS`
- quick correlation: `RM`, `LSTAT` vs `MEDV`


---

# Pocket Recap

| Move | One line |
|------|----------|
| Load | `sep=r'\s+'`, pass column names |
| Y | `MEDV` in $1000s |
| EDA | look before `fit` |
| `CHAS` | 0/1 categorical |
| Hue | second variable as color |
| Barplot | category vs **mean** |
| KDE | smooth density, `hue` compares groups |
| Scatter | two numbers, optional color = Y |

Three things:

1. No header in `housing.csv` — names are on you.  
2. `CHAS` is rare (35 rows). Small groups lie quieter in plots.  
3. EDA is nosy. Nosy is the job.


---

# Practice — your turn

Use `df`. Already loaded.

## P1 — Screening (Easy)
Print shape, `df.info()`, and confirm zero missing.

## P2 — Categorical (Easy)
Countplot of `CHAS`. Print `value_counts(normalize=True)`.

## P3 — Numerical (Easy)
Histogram of `RM` and `MEDV` on one figure (two subplots).

## P4 — Hue (Medium)
Create `Pricey` (above median `MEDV`). Countplot `CHAS` with `hue=Pricey`. Print crosstab.

## P5 — Barplot (Medium)
Mean `MEDV` by `CHAS`. One sentence: river towns higher or lower on average?

## P6 — KDE (Medium)
KDE of `LSTAT` with `hue=CHAS`.

## P7 — Scatter (Medium)
Scatter `RM` vs `MEDV`, `hue=CHAS`. Print `groupby('CHAS')['MEDV'].mean()`.


In [ ]:
# P1


In [ ]:
# P2


In [ ]:
# P3


In [ ]:
# P4


In [ ]:
# P5


In [ ]:
# P6


In [ ]:
# P7


That is Boston housing EDA.

You looked. You plotted. You did not `fit` on row index.

Put `housing.csv` beside this notebook. Kaggle link in the first cell if you need a fresh copy.

See you at prediction week. Leave `inplace=True` alone.
